# Интеллектуальная система генерации стихов

Финальный Colab-блокнот для учебного проекта.

В проекте есть два подхода:

1. Цепи Маркова.
2. LSTM с простым механизмом attention.

Блокнот рассчитан на запуск сверху вниз. Сначала мы подготавливаем проект и датасет, потом запускаем Markov-часть, затем LSTM-часть.

## 1. Настройки

Если проект лежит в другом репозитории, поменяйте `REPO_URL`.

In [ ]:
import os

REPO_URL = "https://github.com/morganizwd/poetry.git"
PROJECT_DIR = "/content/poetry"

DATASET_JSON_URL = "https://raw.githubusercontent.com/Koziev/Rifma/main/rifma_dataset.json"
DATASET_JSON_PATH = "/content/rifma_dataset.json"
DATASET_TXT_PATH = "/content/poems_clean.txt"

# Стартовая строка задаёт теме хотя бы небольшой смысловой якорь.
START_LINE = "Я вышел ночью в сад пустой"

# Уменьшаем шум от TensorFlow в логах.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("Настройки готовы")

## 2. Установка библиотек

Для цепей Маркова нужна библиотека `markovify`. TensorFlow обычно уже установлен в Colab.

In [ ]:
import importlib.util
import subprocess
import sys

def install_if_missing(module_name, pip_name=None):
    pip_name = pip_name or module_name
    if importlib.util.find_spec(module_name) is None:
        print(f"Устанавливаем {pip_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
    else:
        print(f"{module_name} уже установлен")

install_if_missing("markovify")
install_if_missing("tensorflow")

print("Библиотеки готовы")

## 3. Загрузка проекта

Если папка `/content/poetry` уже есть, блокнот использует её. Если папки нет, проект будет скачан из GitHub.

In [ ]:
from pathlib import Path
import subprocess

project_path = Path(PROJECT_DIR)

if project_path.exists():
    print("Папка проекта уже есть:", PROJECT_DIR)
else:
    print("Скачиваем проект...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

print("Содержимое проекта:")
for item in sorted(project_path.iterdir()):
    print("-", item.name)

## 4. Загрузка датасета

Для примера используется открытый датасет RIFMA. Мы скачиваем JSON и преобразуем его в простой текстовый файл `poems_clean.txt`, который нужен текущему проекту.

In [ ]:
import urllib.request

if Path(DATASET_JSON_PATH).exists():
    print("JSON-датасет уже скачан:", DATASET_JSON_PATH)
else:
    print("Скачиваем датасет...")
    urllib.request.urlretrieve(DATASET_JSON_URL, DATASET_JSON_PATH)

print("Файл датасета:", DATASET_JSON_PATH)
print("Размер файла:", Path(DATASET_JSON_PATH).stat().st_size, "байт")

## 5. Преобразование датасета в poems_clean.txt

Эта ячейка специально написана устойчиво: она ищет стихотворный текст даже если структура JSON немного отличается.

In [ ]:
import json
import re

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Тип data:", type(data))
if hasattr(data, "__len__"):
    print("Размер data:", len(data))

if isinstance(data, dict):
    print("Ключи верхнего уровня:", list(data.keys())[:20])
    for key in ["data", "dataset", "items", "poems", "samples"]:
        if key in data and isinstance(data[key], list):
            data = data[key]
            print("Используем список из ключа:", key)
            break

if isinstance(data, list) and data:
    print("Пример первого элемента:")
    print(repr(data[0])[:1000])

def count_good_lines(text):
    count = 0
    for line in text.splitlines():
        words = re.findall(r"[А-Яа-яЁё]+", line)
        if len(words) >= 2:
            count += 1
    return count

def extract_text(item):
    candidates = []

    def walk(obj):
        if isinstance(obj, str):
            candidates.append(obj)
        elif isinstance(obj, dict):
            for value in obj.values():
                walk(value)
        elif isinstance(obj, (list, tuple)):
            for value in obj:
                walk(value)

    walk(item)

    if not candidates:
        return ""

    candidates.sort(
        key=lambda s: (count_good_lines(s), len(re.findall(r"[А-Яа-яЁё]", s))),
        reverse=True,
    )
    return candidates[0]

def remove_russian_accent_marks(text):
    # Убираем только знаки ударения. Букву й не трогаем.
    return text.replace("\u0301", "").replace("\u0300", "")

poems = []

for item in data:
    text = extract_text(item)
    if not text:
        continue

    text = remove_russian_accent_marks(text)

    clean_lines = []

    for line in text.splitlines():
        line = line.strip()
        line = re.sub(r"[^А-Яа-яЁё\-\s]", "", line)
        line = re.sub(r"\s+", " ", line).strip()

        words = re.findall(r"[А-Яа-яЁё]+", line)
        if len(words) >= 2:
            clean_lines.append(line)

    if len(clean_lines) >= 2:
        poems.append("\n".join(clean_lines))

with open(DATASET_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("\n\n".join(poems))

print("Готово. Количество стихотворных фрагментов:", len(poems))

if len(poems) == 0:
    raise ValueError("Не удалось извлечь стихи из JSON. Нужно проверить структуру датасета.")

print("\nПервые строки poems_clean.txt:")
with open(DATASET_TXT_PATH, "r", encoding="utf-8") as f:
    for index, line in enumerate(f):
        if index >= 20:
            break
        print(line.rstrip())

## 6. Копирование датасета в папки проекта

In [ ]:
import shutil

markov_dataset_path = Path(PROJECT_DIR) / "Markov_chain_2" / "poems_clean.txt"
lstm_dataset_path = Path(PROJECT_DIR) / "LSTM_generation_2" / "poems_clean.txt"

shutil.copy(DATASET_TXT_PATH, markov_dataset_path)
shutil.copy(DATASET_TXT_PATH, lstm_dataset_path)

print("Датасет скопирован:")
print(markov_dataset_path)
print(lstm_dataset_path)

## 7. Быстрая проверка файлов проекта

Проверяем, что Python-файлы не содержат синтаксических ошибок.

In [ ]:
!python -m py_compile /content/poetry/Markov_chain_2/Markov_chain.py
!python -m py_compile /content/poetry/LSTM_generation_2/lstm_generation.py

print("Синтаксис проверен")

## 8. Генерация через цепи Маркова

Эта часть обучается быстро.

In [ ]:
import sys

markov_dir = Path(PROJECT_DIR) / "Markov_chain_2"
sys.path.insert(0, str(markov_dir))
os.chdir(markov_dir)

from Markov_chain import RhymingPoetryGenerator

markov_generator = RhymingPoetryGenerator(model_name="poet_markov")

if markov_generator.load_and_train("poems_clean.txt", state_size=2):
    markov_poem = markov_generator.generate_poem(
        lines=8,
        rhyme_scheme="AABB",
        start_line=START_LINE,
    )
    markov_generator.display_poem(markov_poem)
    markov_generator.display_metrics()
else:
    print("Markov-часть не обучилась. Проверьте poems_clean.txt")

## 9. LSTM: быстрая проверка обучения

Сначала запускаем маленькую проверочную модель. Это нужно, чтобы убедиться, что обучение реально идёт.

Если видите строки `batch ...` и `Epoch ... Loss ...`, значит обучение работает.

In [ ]:
lstm_dir = Path(PROJECT_DIR) / "LSTM_generation_2"
sys.path.insert(0, str(lstm_dir))
os.chdir(lstm_dir)

from lstm_generation import LSTMRhymingPoetryGenerator

quick_lstm = LSTMRhymingPoetryGenerator(model_name="poet_quick")

quick_ok = quick_lstm.load_and_train(
    "poems_clean.txt",
    epochs=1,
    batch_size=16,
    max_vocab_size=3000,
    max_training_lines=2000,
)

if quick_ok:
    quick_poem = quick_lstm.generate_poem(
        lines=4,
        rhyme_scheme="AABB",
        start_line=START_LINE,
        temperature=0.65,
    )
    quick_lstm.display_poem(quick_poem)
    quick_lstm.display_metrics()
else:
    print("LSTM не обучилась. Проверьте poems_clean.txt")

## 10. LSTM: основной запуск

Запускайте эту ячейку после быстрой проверки. Она обучает модель дольше, использует больше слов и берёт стартовую строку как смысловой якорь.

LSTM теперь обучается на обычном порядке слов и видит границы строк через токен `<line>`. Для рифмы она генерирует несколько кандидатов и выбирает строку с более подходящим последним словом.

Если нужно быстрее, уменьшите `epochs` до `3`, `5` или `10`.

In [ ]:
full_lstm = LSTMRhymingPoetryGenerator(model_name="poet")

full_ok = full_lstm.load_and_train(
    "poems_clean.txt",
    epochs=20,
    batch_size=32,
    max_vocab_size=12000,
    max_training_lines=20000,
)

if full_ok:
    full_poem = full_lstm.generate_poem(
        lines=8,
        rhyme_scheme="AABB",
        start_line=START_LINE,
        temperature=0.65,
    )
    full_lstm.display_poem(full_poem)
    full_lstm.display_metrics()
else:
    print("Основная LSTM-модель не обучилась. Проверьте poems_clean.txt")

## 11. Что считать успешным результатом

Проект работает корректно, если:

- Markov-часть выводит статистику датасета, стихотворение и метрики.
- Быстрая LSTM-проверка выводит `batch ...`, затем `Epoch 1/1, Loss ...`.
- Основная LSTM-модель после обучения выводит стихотворение и метрики.

Предупреждения TensorFlow про `cuFFT`, `cuDNN`, `cuBLAS` в Colab обычно не являются ошибкой.